# EDA & Cleaning: FAERS Warfarin Adverse Event Reports

Source: `faers_warfarin/faers_warfarin_skip*.xlsx` (21 paginated batches). FDA Adverse Event Reporting System spontaneous reports mentioning warfarin.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_columns", 100)
DATA_DIR = Path("..").resolve()
DECISIONS_PATH = Path("dataset_decisions.json")

def record_decision(name, verdict, reason, n_rows=None, n_cols=None, saved_to=None):
    decisions = {}
    if DECISIONS_PATH.exists():
        decisions = json.loads(DECISIONS_PATH.read_text())
    decisions[name] = {
        "verdict": verdict,
        "reason": reason,
        "n_rows": n_rows,
        "n_cols": n_cols,
        "saved_to": saved_to,
    }
    DECISIONS_PATH.write_text(json.dumps(decisions, indent=2))
    print(f"Recorded decision for '{name}': {verdict}")


In [ ]:
import glob
files = sorted(glob.glob(str(DATA_DIR / "not_usable/faers_warfarin/*.xlsx")))
print(len(files), "files")
dfs = [pd.read_excel(f) for f in files]
faers = pd.concat(dfs, ignore_index=True).drop_duplicates(subset="safetyreportid")
print(faers.shape)
faers.head(3)

In [ ]:
faers.isna().mean().sort_values(ascending=False).to_frame("pct_missing").head(15)

## Why this doesn't fit the modeling pipeline

FAERS is spontaneous adverse-event reporting: no genotype, no INR, no maintenance dose, just serious/death/hospitalization flags per report. It cannot supply a supervised regression/classification target for dose or INR prediction, and reports are not linked to the IWPC/MIMIC patients.

## Verdict

**DROP from the modeling pipeline.** Deduplicated table saved for possible narrative use (for example a safety-outcomes discussion section in the review), but excluded from dataset selection for the predictive model.

In [ ]:
out_path = Path("cleaned/faers_warfarin_deduped.csv")
faers.to_csv(out_path, index=False)
record_decision(
    "faers_warfarin",
    verdict="DROP (context only)",
    reason="Spontaneous adverse-event reports: no genotype/INR/dose fields, not patient-linked to other cohorts. Not usable as a supervised training source; kept only for narrative safety discussion.",
    n_rows=faers.shape[0], n_cols=faers.shape[1], saved_to=str(out_path),
)